# 15.2 能否準備多組 FFN？

# 第 15 章：從 Dense 到 Mixture of Experts（MoE）

前置：第4章Dense FFN。把token想成訂單，experts像不同廚師，router像分單員。這個譬喻只描述分工：專家不會天生知道中文或數學，專長需由資料與訓練觀察。實作是dropless小型MoE，不需分散式系統。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：四位師傅都在，這張訂單只請兩位動手**

例中只派給A與C；未選中師傅的權重仍佔儲存空間。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/moe.svg")))

**先想一想：**把同一module放列表三次，是三位獨立廚師嗎？

先準備多套獨立FFN。即使shape相同，它們有不同權重；沒有router時還不知道每筆訂單送哪一位。

**把直覺寫成數學：**E個expert的參數約E×單expert參數。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.modern import DenseFFN

experts = nn.ModuleList([DenseFFN(4, hidden=8) for _ in range(3)])
x = torch.ones(1, 4)
print([e(x).detach().tolist() for e in experts])
assert experts[0].up.weight is not experts[1].up.weight

**如何讀結果：**要新建每一個expert；複製引用會讓參數共用，不是多專家。

**只改一件事：**只故意共用同一module，檢查is身份。
